In [ ]:
!pip3 install numpy

In [ ]:
!pip3 install opencv-python

In [ ]:
!pip3 install ultralytics

In [ ]:
class VFAIConfig:
    def __init__(self):
        self._threshold = 0.8
        self._model = None
        self._verbose = False
        
        self._source = None
        self._source_fps = 0.0

        self._target_width = 0
        self._target_height = 0
        self._target_fps = 0

        self._drop_frames_to_match_in_fps = False
    
    def get_threshold(self):
        return self._threshold
    
    def get_model(self):
        return self._model
    
    def get_verbosity(self):
        return self._verbose
    
    def get_source(self):
        return self._source

    def get_source_fps(self):
        return self._source_fps
    
    def _get_target_width(self):
        return self._target_width
    
    def _get_target_height(self):
        return self._target_height
    
    def _get_target_fps(self):
        return self._target_fps
    
    def _is_drop_frames_to_match_in_fps(self):
        return self._drop_frames_to_match_in_fps

In [ ]:
class VFAIFrame:
    def __init__(self, data=None, since_start=None, epoch=None, metadata={}) -> None:
        self._data = data
        self._since_start = since_start
        self._epoch = epoch
        self._metadata = metadata

In [ ]:
class VFAIQueue:
    def __init__(self, size):
        self.size = size
        self.queue = [None] * self.size
        self.rear = self.front = -1

    def enqueue(self, element):
        if self.front == (self.rear + 1) % self.size:
            print('Queue is full')
            return None
        if self.front == -1:
            self.front = 0
        self.rear = (self.rear+1) % self.size
        self.queue[self.rear] = element
    
    def dequeue(self):
        if self.front == -1:
            # print('Queue is empty')
            return None
        element = self.queue[self.front]
        self.queue[self.front] = None
        if self.front == self.rear:
            self.front = self.rear = -1
        else:
            self.front = (self.front+1) % self.size
        return element
    
    # Function to display the elements of the queue
    def displayQueue(self):
        if self.front == -1:
            print("Queue is Empty")
            return
        print("Elements in the Circular Queue are: ")
        if self.rear >= self.front:
            for i in range(self.front, self.rear + 1):
                print(self.queue[i], end=" ")
        else:
            for i in range(self.front, self.size):
                print(self.queue[i], end=" ")
            for i in range(0, self.rear + 1):
                print(self.queue[i], end=" ")
        print()

In [ ]:
class VFAIStat:
    def __init__(self):
        self._in = 0
        self._ok = 0
        self._err = 0
        self._invalid = 0
    
    def add_in(self):
        self._in += 1
    
    def get_in(self):
        return self._in
    
    def add_ok(self):
        self._ok += 1
    
    def get_ok(self):
        return self._ok
    
    def add_err(self):
        self._err += 1
    
    def get_err(self):
        return self._err
    
    def add_invalid(self):
        self._invalid += 1
    
    def get_invalid(self):
        return self._invalid

In [ ]:
import cv2
import threading
import time

class VFAISource:
    def __init__(self, config: VFAIConfig, qsize=10, name="Worker-VFAISource"):
        self.name = name
        self._stop_event = threading.Event()
        self._thread = None

        self._qsize = qsize
        self._cap = None
        self._config = config
        self._stat = VFAIStat()
        self._frame_queue = VFAIQueue(qsize)

    def start(self):
        if self._thread and self._thread.is_alive():
            return
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._run, name=self.name)
        self._thread.start()

    def _run(self):
        try:
            self.run()
        finally:
            self.cleanup()

    def run(self):
        self._init_source()
        try:
            start = time.time()
            while not self._stop_event.is_set():
                self._stat.add_in()
                ret, frame = self._cap.read()
                if not ret:
                    self._stat.add_err()
                    self._deinit_source()
                    self._init_source()
                    continue
                self._stat.add_ok()
                frame = cv2.resize(frame, (self._new_w, self._new_h))
                vframe = VFAIFrame(data=frame,
                                   since_start=time.time() - start,
                                   epoch=time.time()
                                   )
                self._frame_queue.enqueue(vframe)
        finally:
            self._deinit_source()
            pass

    def stop(self):
        self._stop_event.set()
        if self._thread:
            self._thread.join()

    def should_stop(self):
        return self._stop_event.is_set()

    def cleanup(self):
        self._deinit_source()

    def _deinit_source(self):
        if self._cap:
            self._cap.release()
            print('Source released.')
            self._cap = None
    
    def _init_source(self):
        print('Initializing source...')
        self._deinit_source()
        self._cap = cv2.VideoCapture(self._config.get_source())
        self._src_width = self._cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        self._src_height = self._cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
        fps = self._cap.get(cv2.CAP_PROP_FPS)
        self._src_fps = int(round(fps)) if fps > 0 else 30  # fallback
        self._src_aspect_ratio = self._src_width / self._src_height
        print(f"Source frame dimensions: {self._src_width} x {self._src_height}, "
              f"Aspect ratio: {self._src_aspect_ratio}, FPS: {self._src_fps}")

        scale = min(self._config._get_target_width() / self._src_width,
                    self._config._get_target_height() / self._src_height)
        self._new_w = int(self._src_width * scale)
        self._new_h = int(self._src_height * scale)
        print(f"New frame dimensions: {self._new_w} x {self._new_h}")

        self._config._source_fps = self._src_fps
        
    
    def get_frame(self):
        try:
            frame = self._frame_queue.dequeue()
            return frame
        finally:
            pass

In [ ]:
import numpy as np
from ultralytics import YOLO

class VFAIDetector:
    def __init__(self, config: VFAIConfig):
        self._config = config
        self._stat = VFAIStat()
        self._model = YOLO(self._config.get_model())
        self._warmup()
    
    def _warmup(self):
        dummy = np.zeros((256, 256, 3), dtype=np.uint8)
        for _ in range(5):
            self._model(dummy, verbose=False)

    def detect(self, frame, threshold=None):
        self._stat.add_in()
        results = self._model(frame,
                              verbose=self._config.get_verbosity(),
                              conf=self._config.get_threshold() if threshold is None else threshold)
        if len(results[0].boxes):
            self._stat.add_ok()
        else:
            self._stat.add_err()
        return results

In [ ]:
import threading
import time
import cv2

class VFAIEngine:
    def __init__(self, config: VFAIConfig, name="Worker-VFAIEngine"):
        self.name = name
        self._stop_event = threading.Event()
        self._thread = None

        self._config = config
        self._stat = VFAIStat()
        self._source = VFAISource(config=self._config, qsize=360000)
        self._detector = VFAIDetector(config=self._config)

    def start(self):
        if self._thread and self._thread.is_alive():
            return
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._run, name=self.name)
        self._thread.start()

    def _run(self):
        try:
            self.run()
        finally:
            self.cleanup()

    def run(self):
        self._source.start()

        # objects = {}
        # occurances = {}

        detection_id = 0
        # object_id = 0

        while not self._stop_event.is_set():
            vframe = self._source.get_frame()
            if vframe is None:
                time.sleep(1/1000)
                continue
            detection_id += 1
            
            # cv2.putText(frame, f"Something: ", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)

            # perform object detection
            detection_startT = time.time()
            results = self._detector.detect(frame=vframe._data)
            detection_endT = time.time()
            if results is None or len(results) == 0:
                continue
            r = results[0]
            boxes = r.boxes
            if len(boxes) == 0:
                continue

            # object_id = 0
            nowt = time.time()
            elapsed = nowt-vframe._epoch

            to_drop = 0
            if self._config._is_drop_frames_to_match_in_fps():
                to_drop = int((elapsed * 1000)/(1000/self._config.get_source_fps()))

            print(f'captured at {vframe._epoch:.0f}, detected at {nowt:.0f} '
                  f'Diff {elapsed:.0f}s, '
                  f'Inference Time {(detection_endT-detection_startT):.2f}s, '
                  f'Source FPS: {self._config.get_source_fps()}, '
                  f'to_drop: {to_drop} frames, '
                  )
            
            # to_drop -= int(to_drop*0.20)
            if self._config._is_drop_frames_to_match_in_fps():
                for _ in range(to_drop):
                    self._source.get_frame()
                print(f'Dropped {to_drop} frames')

        
            # for box in boxes:
            #     frame = vframe._data
            #     box_count += 1
            #     cls_id = int(box.cls[0])                # class index
            #     conf = float(box.conf[0])               # confidence
            #     name = r.names[cls_id]                  # class name
            #     x1, y1, x2, y2 = map(int, box.xyxy[0])  # bounding box coordinates

            #     if True or conf > threshold:
            #         occurances[id] = {
            #             'id': id,
            #             'name': name,
            #             'class_id': cls_id,
            #             'confidence': f'{conf:.2f}',
            #             "since_start": f'{vframe._since_start:.2f}',
            #             "epoch": f'{vframe._epoch:.0f}',
            #             "detected_at": f'{nowt:.0f}'
            #         }
            #         id += 1

                    # cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    # label = f"{name} {conf:.2f}"
                    # cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                    # cv2.imwrite(f"output_{frame_count}_box_{box_count}.jpg", frame)
                    # cv2.imshow("YOLO", frame)

            # annotated = results[0].plot()
            # cv2.imwrite(f"output_{frame_count}.jpg", annotated)
            # cv2.imshow("YOLO", annotated)

            # cv2.imshow("Frame", frame)
            # if cv2.waitKey(1) == ord('q'):
            #     break

        # print("Final object counts:", objects)
        # print("Occurances:", occurances)
        # cv2.destroyAllWindows()
        print("Destroyed")

    def stop(self):
        self._stop_event.set()
        if self._thread:
            self._thread.join()

    def should_stop(self):
        return self._stop_event.is_set()

    def cleanup(self):
        self._source.stop()
    

In [ ]:
import signal
import sys

def main():
    engine = None
    def shutdown(signum, frame):
        print(f"Signal {signum} received, shutting down...")
        engine.stop()
        sys.exit(0)

    signal.signal(signal.SIGINT, shutdown)   # Ctrl+C
    signal.signal(signal.SIGTERM, shutdown)  # kill command

    config = VFAIConfig()
    config._source = 'C:/Users/soura/Downloads/15192700-sd_240_426_30fps.mp4'
    config._source = 'C:/Users/soura/Downloads/14778909_640_360_60fps.mp4'
    # config._source = 'C:/Users/soura/Downloads/istockphoto-1162603138-640_adpp_is.mp4'
    config._target_height = 256
    config._target_width = 256
    config._target_fps = 10
    config._model = "yolov8n.pt"
    config._threshold = 0.3
    config._drop_frames_to_match_in_fps = True
    
    engine = VFAIEngine(config=config)

    engine.start()
    while True:
        pass

if __name__ == "__main__":
    main()